#  Arm Assembly Primer : `if` statements
# 1. Conditional branches
* So far, instructions were executed one after another, from top to bottom
* An `if` statement needs to skip some instructions depending on a condition, which is done by a <font color="blue">conditional branch</font> instruction: "jump to a label if a condition holds; otherwise continue with the next instruction"


In [ ]:
%%writefile branch.c
long report_neg(long x);   /* defined elsewhere */
long report_pos(long x);   /* defined elsewhere */

long check_lt(long x, long y) {
  if (x < y) {
    x = report_neg(x) + 1;
  }
  return x * y + 10;
}

long if_else(long x, long y) {
  long r;
  if (x < y) {
    r = report_neg(x) + 1;
  } else {
    r = report_pos(y) + 2;
  }
  return r + 10;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S branch.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' branch.s

* You will see something like this

```
check_lt:
	stp	x29, x30, [sp, #-32]!
	str	x19, [sp, #16]      // save x19 (see below)
	mov	x29, sp
	mov	x19, x1             // keep y in x19
	cmp	x0, x1              // compare x and y
	b.ge	.LBB0_2             // if x >= y, jump to .LBB0_2 (skip the then-part)
	bl	report_neg          // (x < y) the then-part: x = report_neg(x) + 1
	add	x0, x0, #1
.LBB0_2:                          // both paths come here
	mov	x8, #10
	madd	x0, x0, x19, x8     // return x * y + 10
	ldr	x19, [sp, #16]      // restore x19
	ldp	x29, x30, [sp], #32
	ret
if_else:
	stp	x29, x30, [sp, #-16]!
	mov	x29, sp
	cmp	x0, x1              // compare x and y
	b.ge	.LBB1_2             // if x >= y, jump to .LBB1_2 (the else-part)
	bl	report_neg          // (x < y) the then-part: r = report_neg(x) + 1
	add	x8, x0, #1
	b	.LBB1_3             // skip the else-part
.LBB1_2:
	mov	x0, x1              // (x >= y) the else-part: r = report_pos(y) + 2
	bl	report_pos
	add	x8, x0, #2
.LBB1_3:                          // both paths come here
	add	x0, x8, #10         // return r + 10
	ldp	x29, x30, [sp], #16
	ret
```

* <font color="blue">`cmp x0, x1`</font> compares `x0` and `x1` and records the result in special 1-bit registers called the _condition flags_ (whether the difference `x0 - x1` is zero, negative, ...)
* <font color="blue">`b.ge .LBB0_2`</font> (_branch if greater than or equal_) jumps to `.LBB0_2` if the flags say `x0 >= x1`; otherwise the next instruction is executed
* Note that the condition is the _opposite_ of the one in C: "if `x >= y`, jump over the then-part"
* In `if_else`, the then-part ends with an unconditional branch (`b .LBB1_3`) that jumps over the else-part, so that exactly one of the two parts is executed; both paths meet again at `.LBB1_3`, where the code after the `if` statement (`return r + 10`) begins
* The general shape of an `if` statement is therefore

```
	compare
	branch to ELSE if the condition does NOT hold
	(then-part)
	b    END
ELSE:
	(else-part)
END:
	(the code after the if statement)
```

* In `check_lt`, `y` is needed after the call to `report_neg` (in `x * y`), but `x1`, which held `y`, may be overwritten by `report_neg`. So `y` is copied into `x19` before the call. `x19` -- `x28` are registers that a called function must leave intact (if it uses them, it saves and restores them, as `check_lt` itself does with `str x19` and `ldr x19`)
* Labels starting with `.L` (`.LBB0_2`) are made up by the compiler; they mark places to jump to inside a function
* The control flow is easier to see in a picture; each box is a _basic block_ (instructions executed straight through, from a label to a branch), and each arrow is a possible jump (T = the branch is taken, F = not taken, i.e., falls through to the next instruction)


In [ ]:
import vis_asm
vis_asm.vis_asm_file("branch.s")

* `check_lt` has a diamond with one empty side: the then-part is either executed or skipped, and both paths reach the block computing `x * y + 10`
* `if_else` has a full diamond: from the `cmp`/`b.ge` block, one path goes through the then-part and the other through the else-part, and they meet again at the block computing `r + 10`
* Conditions you will often see

<div>

| condition | after `cmp a, b` (signed) | condition | after `cmp a, b` (unsigned) |
|-----------|---------------------------|-----------|-----------------------------|
| `eq` | `a == b` | `eq` | `a == b` |
| `ne` | `a != b` | `ne` | `a != b` |
| `lt` | `a < b` | `lo` | `a < b` |
| `le` | `a <= b` | `ls` | `a <= b` |
| `gt` | `a > b` | `hi` | `a > b` |
| `ge` | `a >= b` | `hs` | `a >= b` |

</div>

* Some branches combine the test and the jump: `cbz x0, L` (jump if `x0` is zero), `cbnz x0, L` (if not zero), `tbnz x0, #63, L` (if bit 63 of `x0`, the sign bit, is 1, i.e., if `x0 < 0`)
# 2. Conditional selects
* An `if` statement does not always become a branch


In [ ]:
%%writefile csel.c
void report(long x);   /* defined elsewhere */

long max_(long a, long b) {
  if (a > b) {
    return a;
  } else {
    return b;
  }
}

void report_smaller(long x, long y) {
  if (x < y) {
    report(x);
  } else {
    report(y + 100);
  }
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S csel.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' csel.s

* You will see something like this

```
max_:
	cmp	x0, x1
	csel	x0, x0, x1, gt   // x0 = (a > b) ? x0 : x1
	ret
report_smaller:
	add	x8, x1, #100       // x8 = y + 100
	cmp	x0, x1
	csel	x0, x0, x8, lt   // x0 = (x < y) ? x : y + 100
	b	report
```

* Visualizing it shows no diamonds at all; each function is a single box, executed straight through


In [ ]:
import vis_asm
vis_asm.vis_asm_file("csel.s")

* <font color="blue">`csel x0, x0, x1, gt`</font> (_conditional select_) sets `x0` to `x0` if the flags say "greater than", and to `x1` otherwise; there is no jump
* `report_smaller` computes `y + 100` regardless of the condition, selects the argument with `csel`, and calls `report` in a single place
* A branch the CPU cannot predict well is costly (the CPU executes instructions ahead of time, guessing which way a branch goes, and has to throw away the work when the guess is wrong), so compilers turn an `if` into `csel` when both sides are cheap to compute and have no side effects
* (On Intel/AMD CPUs, the corresponding instruction is `cmov`, conditional move)

# 3. Summary
* an `if` statement generally becomes a compare (`cmp`) and a conditional branch (`b.lt`, `b.ge`, ...) jumping over the then-part (or to the else-part)
* when both sides are cheap, it may become a conditional select (`csel`) without any branch